# 📈 실험 결과 비교 분석

이 노트북은 MLflow에 기록된 모든 평가 결과를 불러와 비교합니다.

## 비교 축

| 축 | 변형 |
|---|------|
| **모델** | Base vs LoRA vs OSFT |
| **지식 접근** | no_knowledge vs RAG |
| **모드** | simple_rag vs agent_rag |

## 비교 지표

- **태스크 성공률** (τ 에피소드): 공식 pass^k
- **진단 정확도**: 오프라인 홀드아웃 결과 (실험실 파생)
- **보존 델타**: ARC-Challenge 정확도 변화 (retention_delta_pp)
- **지연 시간**: 에피소드 평균 응답 시간
- **실패 분석**: 유형별 오류 분류

> ⚠️ 이 비교는 실제 MLflow 결과를 기반으로 합니다.  
> 가공된 성능 수치를 삽입하지 않습니다.

In [ ]:
"""환경 부트스트랩 — local과 workbench 모두 지원."""

import subprocess, sys
from pathlib import Path

# 프로젝트 루트 탐색 (노트북 위치 기준)
_nb_dir = Path.cwd()
_project_root = _nb_dir
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / "pyproject.toml").exists():
        _project_root = _p
        break

# 패키지 설치 확인 및 자동 설치
try:
    import rhoai_model_training_lab  # noqa: F401
    print("✅ rhoai_model_training_lab 패키지 확인됨")
except ImportError:
    print("📦 패키지 설치 중... (최초 1회)")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-e", str(_project_root)],
        stdout=subprocess.DEVNULL,
    )
    print("✅ 설치 완료 — 커널 재시작이 필요할 수 있습니다.")


In [ ]:
"""Load all evaluation runs from MLflow."""

import os
import json
from pathlib import Path

import pandas as pd

from rhoai_model_training_lab.config import load_env, load_eval_config, PROJECT_ROOT

load_env()

eval_config = load_eval_config()
mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
experiment_name = os.environ.get("MLFLOW_EXPERIMENT_EVAL", "rhoai-model-training-lab-evaluation")

runs_df = None

if mlflow_uri:
    try:
        import mlflow

        mlflow.set_tracking_uri(mlflow_uri)
        experiment = mlflow.get_experiment_by_name(experiment_name)

        if experiment:
            runs = mlflow.search_runs(
                experiment_ids=[experiment.experiment_id],
                order_by=["start_time DESC"],
            )
            runs_df = runs
            print(f"✅ MLflow에서 {len(runs)}개 실행 로드 완료")
            print(f"   실험: {experiment_name}")
            print(f"   URI: {mlflow_uri}")
        else:
            print(f"⚠️  실험 '{experiment_name}'을 찾을 수 없습니다.")
    except Exception as exc:
        print(f"❌ MLflow 로드 실패: {exc}")
else:
    print("⚠️  MLFLOW_TRACKING_URI 미설정")

# Fallback: load from local result files
if runs_df is None or len(runs_df) == 0:
    print("\n📂 로컬 결과 파일에서 로드 시도...")
    results_dir = PROJECT_ROOT / eval_config["general"]["output_dir"]
    local_results = []

    for rf in sorted(results_dir.glob("**/*.json")):
        try:
            with open(rf) as f:
                data = json.load(f)
            if isinstance(data, dict):
                data["_source_file"] = str(rf.relative_to(results_dir))
                local_results.append(data)
        except Exception:
            continue

    if local_results:
        runs_df = pd.DataFrame(local_results)
        print(f"  {len(local_results)}개 로컬 결과 파일 로드 완료")
    else:
        print("  ⚠️  결과 파일을 찾을 수 없습니다.")
        print("  07_evaluate.ipynb를 먼저 실행하세요.")

if runs_df is not None and len(runs_df) > 0:
    print(f"\n총 {len(runs_df)}개 실행 데이터 사용 가능")

In [ ]:
"""Build comparison table: Base vs LoRA vs OSFT × knowledge access."""

from rich.console import Console
from rich.table import Table

console = Console()

print("=" * 70)
print("📊 실험 비교 테이블")
print("=" * 70)

# Define expected variants
expected_variants = [
    {"variant": "base_no_knowledge", "model": "Base", "knowledge": "No KB"},
    {"variant": "base_rag", "model": "Base", "knowledge": "RAG"},
    {"variant": "lora_no_knowledge", "model": "LoRA", "knowledge": "No KB"},
    {"variant": "lora_rag", "model": "LoRA", "knowledge": "RAG"},
    {"variant": "osft_no_knowledge", "model": "OSFT", "knowledge": "No KB"},
    {"variant": "osft_rag", "model": "OSFT", "knowledge": "RAG"},
]

table = Table(title="모델 × 지식 접근 비교", show_header=True)
table.add_column("변형", style="bold")
table.add_column("모델")
table.add_column("지식 접근")
table.add_column("태스크 성공률")
table.add_column("진단 정확도")
table.add_column("보존 Δpp")
table.add_column("평균 지연(s)")
table.add_column("실패 수")

if runs_df is not None and len(runs_df) > 0:
    for ev in expected_variants:
        variant = ev["variant"]
        # Try to find matching run
        metric_cols = [c for c in runs_df.columns if "metric" in c.lower() or variant in str(c).lower()]

        # Extract metrics (adapt to actual MLflow column naming)
        task_sr = "N/A"
        diag_acc = "N/A"
        retention = "N/A"
        latency = "N/A"
        failures = "N/A"

        # Look for matching data in runs
        for col_prefix in ["metrics.", "params."]:
            if f"{col_prefix}variant" in runs_df.columns:
                mask = runs_df[f"{col_prefix}variant"] == variant
                matched = runs_df[mask]
                if len(matched) > 0:
                    row = matched.iloc[0]
                    task_sr = f"{row.get('metrics.task_success_rate', 'N/A')}"
                    diag_acc = f"{row.get('metrics.answer_accuracy', 'N/A')}"
                    retention = f"{row.get('metrics.retention_delta_pp', 'N/A')}"
                    latency = f"{row.get('metrics.mean_wall_time_seconds', 'N/A')}"
                    failures = f"{row.get('metrics.failed', 'N/A')}"

        table.add_row(
            variant, ev["model"], ev["knowledge"],
            task_sr, diag_acc, retention, latency, failures,
        )
else:
    for ev in expected_variants:
        table.add_row(
            ev["variant"], ev["model"], ev["knowledge"],
            "—", "—", "—", "—", "—",
        )

console.print(table)

if runs_df is None or len(runs_df) == 0:
    print("\n⚠️  평가 데이터가 없어 빈 테이블입니다.")
    print("   07_evaluate.ipynb를 먼저 실행하세요.")

In [ ]:
"""Plot task success rates."""

try:
    import matplotlib.pyplot as plt
    import numpy as np

    print("=" * 70)
    print("📊 태스크 성공률 시각화")
    print("=" * 70)

    models = ["Base", "LoRA", "OSFT"]
    no_kb_rates = []
    rag_rates = []

    # Extract data from runs or use placeholders
    for model in models:
        no_kb_key = f"{model.lower()}_no_knowledge"
        rag_key = f"{model.lower()}_rag"

        no_kb_val = 0.0
        rag_val = 0.0

        if runs_df is not None and len(runs_df) > 0:
            for _, row in runs_df.iterrows():
                v = row.get("params.variant", row.get("variant", ""))
                sr = row.get("metrics.task_success_rate", row.get("task_success_rate", None))
                if sr is not None and not pd.isna(sr):
                    if v == no_kb_key:
                        no_kb_val = float(sr)
                    elif v == rag_key:
                        rag_val = float(sr)

        no_kb_rates.append(no_kb_val)
        rag_rates.append(rag_val)

    x = np.arange(len(models))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))
    bars1 = ax.bar(x - width/2, no_kb_rates, width, label="No Knowledge", color="#90CAF9")
    bars2 = ax.bar(x + width/2, rag_rates, width, label="RAG", color="#FF8A65")

    ax.set_ylabel("Task Success Rate")
    ax.set_title("모델 × 지식 접근별 태스크 성공률")
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.legend()
    ax.set_ylim(0, 1.0)
    ax.grid(axis="y", alpha=0.3)

    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f"{height:.2f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.show()

    if all(v == 0 for v in no_kb_rates + rag_rates):
        print("⚠️  모든 성공률이 0입니다 — 평가 데이터가 아직 없는 것 같습니다.")

except ImportError:
    print("matplotlib가 설치되지 않아 시각화를 건너뜁니다.")
    print("pip install matplotlib 을 실행하세요.")

In [ ]:
"""Plot retention delta."""

try:
    import matplotlib.pyplot as plt
    import numpy as np

    print("=" * 70)
    print("📊 보존 델타 (Retention Delta) 시각화")
    print("=" * 70)

    models = ["LoRA", "OSFT"]
    deltas = []
    base_acc = 0.0

    for model in models:
        delta = 0.0
        if runs_df is not None and len(runs_df) > 0:
            for _, row in runs_df.iterrows():
                v = row.get("params.variant", row.get("variant", ""))
                d = row.get("metrics.retention_delta_pp", row.get("retention_delta_pp", None))
                if d is not None and not pd.isna(d) and model.lower() in str(v).lower():
                    delta = float(d)
                ba = row.get("metrics.base_accuracy", row.get("base_accuracy", None))
                if ba is not None and not pd.isna(ba) and "base" in str(v).lower():
                    base_acc = float(ba)
        deltas.append(delta)

    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["#4CAF50" if d >= 0 else "#F44336" for d in deltas]
    bars = ax.bar(models, deltas, color=colors, width=0.5)

    ax.axhline(y=0, color="black", linewidth=0.8, linestyle="-")
    ax.set_ylabel("Retention Delta (pp)")
    ax.set_title(f"ARC-Challenge 보존 델타\n(기본 모델 정확도: {base_acc:.1%})")
    ax.grid(axis="y", alpha=0.3)

    for bar, delta in zip(bars, deltas):
        y_pos = bar.get_height() + (0.3 if delta >= 0 else -0.5)
        ax.text(bar.get_x() + bar.get_width()/2., y_pos,
                f"{delta:+.1f}pp", ha="center", va="bottom" if delta >= 0 else "top")

    plt.tight_layout()
    plt.show()

    print("retention_delta_pp = 100 × (적응 모델 정확도 - 기본 모델 정확도)")
    print("양수 = 기존 능력 향상, 음수 = 기존 능력 저하 (망각)")

except ImportError:
    print("matplotlib가 설치되지 않아 시각화를 건너뜁니다.")

In [ ]:
"""Analyze failures and latency."""

print("=" * 70)
print("🔍 실패 분석 및 지연 시간")
print("=" * 70)

if runs_df is not None and len(runs_df) > 0:
    # Failure analysis
    print("\n--- 실패 유형 분석 ---")
    error_cols = [c for c in runs_df.columns if "error" in c.lower() or "fail" in c.lower()]
    if error_cols:
        for col in error_cols:
            values = runs_df[col].dropna()
            if len(values) > 0:
                print(f"\n  {col}:")
                if values.dtype in ["int64", "float64"]:
                    print(f"    총계: {values.sum():.0f}")
                    print(f"    평균: {values.mean():.1f}")
                else:
                    for v, count in values.value_counts().items():
                        print(f"    {v}: {count}")
    else:
        print("  실패 관련 열을 찾을 수 없습니다.")

    # Latency analysis
    print("\n--- 지연 시간 분석 ---")
    latency_cols = [c for c in runs_df.columns if "latency" in c.lower() or "time" in c.lower()]
    if latency_cols:
        for col in latency_cols:
            values = runs_df[col].dropna()
            if len(values) > 0 and values.dtype in ["int64", "float64"]:
                print(f"\n  {col}:")
                print(f"    평균: {values.mean():.2f}")
                print(f"    중앙값: {values.median():.2f}")
                print(f"    최소: {values.min():.2f}")
                print(f"    최대: {values.max():.2f}")
    else:
        print("  지연 시간 관련 열을 찾을 수 없습니다.")

    # Token usage
    print("\n--- 토큰 사용량 ---")
    token_cols = [c for c in runs_df.columns if "token" in c.lower()]
    if token_cols:
        for col in token_cols:
            values = runs_df[col].dropna()
            if len(values) > 0 and values.dtype in ["int64", "float64"]:
                print(f"  {col}: 평균={values.mean():.0f}, 총합={values.sum():.0f}")
    else:
        print("  토큰 사용량 관련 열을 찾을 수 없습니다.")
else:
    print("⚠️  분석할 데이터가 없습니다.")

In [ ]:
"""Summary and conclusions."""

from rich.panel import Panel

print("=" * 70)
print("📝 요약 및 결론")
print("=" * 70)

summary_text = """
🏦 τ-Knowledge Banking 모델 학습 실험 결과 요약

실험 설정:
  • 기본 모델: Qwen/Qwen3-4B-Instruct-2507
  • 도메인: τ-Knowledge banking_knowledge
  • 학습 방법: LoRA (r=16, α=32) / OSFT (unfreeze=0.25)
  • 평가: 진단(오프라인) + τ 에피소드(온라인) + 보존(ARC-Challenge)
"""

console.print(Panel(summary_text, title="실험 요약", border_style="blue"))

# Conclusions based on available data
print("\n주요 관찰 사항:")
print()

if runs_df is not None and len(runs_df) > 0:
    print("  1. 모델 성능 비교:")
    print("     - 구체적인 수치는 위 테이블과 그래프를 참조하세요.")
    print("     - 소규모 스모크 테스트로는 벤치마크 성능 결론을 내릴 수 없습니다.")
    print()
    print("  2. KB 적응 실험 (kb_adaptation):")
    print("     - 이 실험은 KB를 학습하고 새로운 상황에 적용하는 능력을 측정합니다.")
    print("     - 학습 없는 리더보드 프로토콜과 동등하다고 주장하지 않습니다.")
    print()
    print("  3. 보존 분석:")
    print("     - 단일 벤치마크의 소규모 샘플로는 망각 제거를 입증할 수 없습니다.")
    print("     - OSFT가 LoRA 대비 보존이 우수하다는 인과적 주장은")
    print("       전체 SFT 통제군 없이는 성립하지 않습니다.")
else:
    print("  ⚠️  평가 데이터가 없습니다.")
    print("     07_evaluate.ipynb를 실행하여 평가를 완료하세요.")

print("\n한계 및 주의사항:")
print("  • 합성 단일 턴 진단 정확도 ≠ 공식 τ 멀티 턴 태스크 성공")
print("  • pass^k (일관된 성공) ≠ pass@k (최선 시도)")
print("  • 소규모 모델(4B)의 전체 에이전트 태스크 성능에는 한계가 있습니다")
print("  • 데이터 준비 품질이 학습 효과를 크게 좌우합니다")

print("\n다음 단계:")
print("  • 더 많은 에피소드와 시행으로 통계적 유의성 확보")
print("  • 학습 데이터 품질 개선 (data_preparation/ 노트북 참조)")
print("  • 하이퍼파라미터 튜닝 (개발 데이터 기반)")
print("  • 전체 SFT 통제군 실험 (선택적)")